# MiMo-V2.5 Inference Optimization Toy Experiments

This notebook clones the project in Colab, then runs the companion Python file `mimo_optimization_experiments.py` from the experiment directory.

It uses small PyTorch models and simulators to compare baseline vs optimized versions of several ideas from Xiaomi's MiMo-V2.5 inference blog: hybrid SWA KV cache, length bucketing, cache-affinity scheduling, encoder batching, and MTP-style speculative decode.

For Colab: open this notebook, select a GPU runtime, and run the setup cell. The setup cell will clone the repo separately under `/content/minicpm-o-agent`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/erkamkavak/minicpm-o-agent.git'
REPO_DIR = Path('/content/minicpm-o-agent')
EXPERIMENT_DIR = Path('experiments/mimo_inference_optimizations')
SCRIPT_NAME = 'mimo_optimization_experiments.py'

def running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

if running_in_colab():
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    os.chdir(REPO_DIR / EXPERIMENT_DIR)
else:
    if not Path(SCRIPT_NAME).exists():
        repo_experiment_dir = Path.cwd() / EXPERIMENT_DIR
        if (repo_experiment_dir / SCRIPT_NAME).exists():
            os.chdir(repo_experiment_dir)
        else:
            raise FileNotFoundError(
                f'Could not find {SCRIPT_NAME}. Open the notebook from the repo root '
                f'or from {EXPERIMENT_DIR}.'
            )

script = Path(SCRIPT_NAME)
print('Working directory:', Path.cwd())
print('Experiment script:', script.resolve())

try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
except ModuleNotFoundError:
    print('Installing PyTorch. Restart the runtime if Colab asks you to.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## Quick Run

This should finish quickly and gives the main intuition. On CPU, some optimized kernels may not beat the baseline, but memory/work reductions are still visible. GPU timing usually shows the intended direction more clearly.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--quick',
    '--device', 'auto',
    '--repeats', '5',
    '--warmup', '2',
    '--json', 'results_quick.json',
], check=True)

## Run Individual Experiments

Use this cell while studying one optimization at a time.

In [ ]:
# Change the name to one of:
# hybrid_swa, length_bucketing, scheduler, encoder_batching, mtp
experiment = 'hybrid_swa'
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--experiments', experiment,
    '--device', 'auto',
    '--quick',
    '--repeats', '7',
    '--warmup', '2',
], check=True)

## Long-Context Hybrid SWA Up To 128K

Run only the Hybrid SWA experiment for long contexts. This is a decode-attention/KV-cache toy benchmark, not a real 128K LLM prefill. It is designed to show the KV memory curve safely in Colab.

In [ ]:
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--experiments', 'hybrid_swa',
    '--hybrid-lengths', '4k,8k,16k,32k,64k,128k',
    '--device', 'auto',
    '--repeats', '3',
    '--warmup', '1',
    '--json', 'results_hybrid_128k.json',
], check=True)

## Less Tiny Run

This uses larger toy settings. It is better on a GPU runtime.

In [ ]:
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--device', 'auto',
    '--repeats', '5',
    '--warmup', '2',
    '--json', 'results_full.json',
], check=True)

## Inspect JSON Results

This renders the JSON output as a comparison table. `speedup > 1` means the optimized version is better. For memory or token-work rows, speedup means reduction in required memory/work.

In [ ]:
import json
from pathlib import Path

result_candidates = [
    Path('results_quick.json'),
    Path('results_full.json'),
    Path('results_hybrid_128k.json'),
]
existing = [candidate for candidate in result_candidates if candidate.exists()]
if not existing:
    raise FileNotFoundError('Run a benchmark cell first.')
path = max(existing, key=lambda candidate: candidate.stat().st_mtime)

rows = json.loads(path.read_text())

records = []
for row in rows:
    unit = row['unit']
    records.append({
        'experiment': row['experiment'],
        'case': row['case'],
        'metric': row['metric'],
        'baseline': f"{row['baseline']:.2f} {unit}",
        'optimized': f"{row['optimized']:.2f} {unit}",
        'speedup': f"{row['speedup']:.2f}x",
        'note': row.get('note', ''),
    })

try:
    import pandas as pd
    df = pd.DataFrame(records)
    display(df)
except Exception:
    widths = {key: max(len(key), *(len(str(r[key])) for r in records)) for key in records[0]}
    columns = list(records[0])
    print('  '.join(col.ljust(widths[col]) for col in columns))
    print('  '.join('-' * widths[col] for col in columns))
    for record in records:
        print('  '.join(str(record[col]).ljust(widths[col]) for col in columns))

print(f'Loaded {path}')
print('\nTop speedups:')
for row in sorted(rows, key=lambda r: r['speedup'], reverse=True)[:5]:
    print(f"- {row['experiment']} / {row['metric']} / {row['case']}: {row['speedup']:.2f}x")